# v11 訓練 — Datasets_YOLO26_v5.6

**這份是 v10 的 `train_v10.ipynb` 原封沿用**，送去 Kaggle 之前只改了兩個值：
`RUN`（`v55_v10` → `v5.6_v11`）與 `DATASET_SRC`（指向 v5.6 的 Kaggle Dataset）。
超參數、模型組態、隔離驗證與輸出整理**逐字相同**——這正是 v11 的實驗設計：
只換資料集、其餘一個字都不動，才問得出「v5.6 相對 v5.5 值多少」。

> **補進版控的時間點晚於執行。** 這份檔案原本只在 Kaggle 與本機下載資料夾，
> 2026-09-10 才補進 repo。補的時候把正文與註解裡仍寫著 v10／v5.5 的字樣改成 v11／v5.6
> （逐項見下方「已知落差」），**程式邏輯一行未動**。

### 相對 v10 的唯一改動

| # | 改動 | 依據 |
| --- | --- | --- |
| 1 | 資料集 `v5.5` → **`v5.6`（仍是 9 類）** | 巢狀切分（v5.6 的評估集 ⊇ v5.5 的評估集）＋ 逐類增強 profile。見 `docs/v5.6_結果_資料集分析與驗收.md` |

超參數與 v10 逐項相同：`epochs=100`、`patience=30`、`close_mosaic` 依 epochs 等比推導。

> **`patience=30` 是 v11 事後被檢討掉的地方。** 早停讓 valid 參與了 checkpoint 決策，
> 於是 v11 的 valid 不能與 test 併計，評估精度目標（九類 ±2SE ≤ 0.10）**當輪證明不出來**。
> v11.5 改成 `patience=0` ＋ 固定 70 輪才解決，並成為往後的訓練協定。
> 見 `docs/v11.5_結果_評估精度驗證.md`。

### 判準

v5.6 的評估集是 v5.5 的**超集**（巢狀切分），所以與 v10 之間存在合法的對照方式：
在 v5.5 的評估子集上重跑一次，即 Step.7 產出的 `local_eval/v55_subset_eval.json`。
**直接拿 v11 的整體 mAP 去比 v10 的整體 mAP 是不對的**——評估集不同。
實際結論見 `docs/v11_結果_v5.6首次訓練評估.md`。

### 已知落差（補進版控時修掉的）

| 位置 | 執行當時 | 現在 |
| --- | --- | --- |
| 標題與說明正文 | v10 / v5.5 | v11 / v5.6 |
| `summary` 的 `"dataset"` 欄 | 寫死 `"v5.5"` | `"v5.6"` |
| 註解、`assert` 訊息、`print` 的版本字樣 | v10 | v11 |
| `YAML_PATH` 的暫存檔名 | `yolo26n-p2-v10.yaml` | `yolo26n-p2-v11.yaml` |

⚠ **因為執行當時 `"dataset"` 那行寫死 `"v5.5"`，`Train Output/extracted/eval/summary.json`
的 `dataset` 欄位記成 `v5.5`，但實際跑的是 v5.6**（同檔的 `run` 欄是 `v5.6_v11`，
且 `DATASET_SRC` 指向 v5.6 的 Kaggle Dataset，兩者可以佐證）。
該檔是實驗產出，**沒有手動修改**。v11.5 與 v12s 的同一欄位是正確的。

### 執行前確認

1. `Datasets_YOLO26_v5.6/OutPut/` 已上傳成 Kaggle Dataset，路徑填進下一個 cell 的 `DATASET_SRC`。
2. **Notebook 設定的 Internet 必須開啟**——要抓 `yolo26n.pt`。
3. 建議用 **Save & Run All**，全程無人看顧。

### `best.pt` 不能只看單點——小樣本評估集會直接汙染 checkpoint 選擇

`DetMetrics.fitness()` = `mAP@0.5:0.95`，**跨全部 9 類巨集平均、每類權重相等**
（`ultralytics/utils/metrics.py` 的 `Metric.fitness()`，`w=[0,0,0,1]`）；`EarlyStopping`
用嚴格大於判斷，任一輪 `fitness` 破新高就重設 `best_epoch`、覆寫 `best.pt`，patience 歸零。

這跟訓練時的加權方向相反：訓練 loss 是按框數分配梯度（`v8DetectionLoss` 的
`target_scores_sum` 是整個 batch 加總後才除，框多的類別天生拿到更多梯度），
但 `fitness` 給每個類別**相等的權重（1/9）**——而 `Thrips_Damage` 是全資料集
評估樣本最小的類別（v5.6 的逐類張數與框數見 `docs/v5.6_結果_資料集分析與驗收.md` §2）。
用本專案在 P3 診斷對類似小樣本規模量出的經驗值推算，單輪 AP50-95 的抽樣雜訊約
±0.15–0.25，換算到聚合 `fitness` 約 ±0.017–0.028——是 A0 平台期實測 σ=0.0038
（2σ 門檻 0.0077）的 **2–4 倍**。

**因此**：`best.pt` 有不小機率是被 `Thrips_Damage` 某一輪的抽樣運氣觸發的，不代表
模型真的變好。**判讀一律連同最後 N 輪的平台期平均一起看**（Step.7 已算好存進
`summary.json` 的 `plateau` 欄位），不要只憑 `best.pt` 單點下結論。


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 唯一需要編輯的 cell
# ══════════════════════════════════════════════════════════════════════
RUN     = "v5.6_v11"     # 訓練輸出目錄名稱
EPOCHS  = 100           # v9 實測最佳點在 ep69，160 輪的末段是純過擬合
PATIENCE = 30           # 與 v8／v10 一致。事後檢討：早停讓 valid 參與決策，v11.5 起改為 0

# Datasets_YOLO26_v5.6/OutPut 上傳成 Kaggle Dataset 之後的路徑
DATASET_SRC = "/kaggle/input/datasets/yentsai9183/datasets-yolo26-v5-6"

# 安全閥。STOP_AFTER_EPOCHS=None 代表跑滿 EPOCHS；只有在單場塞不下、
# 需要拆成兩場時才填數字（填了會在該輪乾淨停止，剩下的交給 RESUME.ipynb）。
# ⚠ 填了數字卻忘記跟著 EPOCHS 調整，訓練會在半路停掉而且不會報錯。
STOP_AFTER_EPOCHS = None
DEADLINE_HOURS    = 10.5    # Save & Run All 上限 12 h，留 1.5 h 餘裕

# ── 超參數：與 v9 A0 逐項相同，維持可對照 ──────────────────────────────
HYPERS = dict(
    optimizer="MuSGD", lr0=0.008, lrf=0.01, momentum=0.937, cos_lr=True,
    warmup_epochs=3.0, mosaic=0.7, cls=0.8, dfl=1.5, box=8.0, imgsz=640,
    batch=20, seed=0, cache="ram", workers=4, plots=True, device=0,
)

# close_mosaic 依 epochs 等比推導（v8 是 160 輪關 30 輪），維持排程形狀一致。
# 註：v9 階段 3 觀察到關 mosaic 之後 train_cls 斷崖式下降、但 val 指標完全沒動，
# 亦即它只是讓模型更用力擬合訓練集。此處保留是為了與 v8/v9 的排程可對照，
# 若要驗證它的去留，應該當成獨立的一臂來測，而不是在這裡順手改掉。
CLOSE_MOSAIC = max(1, round(30 * EPOCHS / 160))
print(f"▷ RUN={RUN}  epochs={EPOCHS}  patience={PATIENCE}  close_mosaic={CLOSE_MOSAIC}")

# Step.1 環境

In [ ]:
!nvidia-smi
# 版本釘死：v9 的所有數字都在 8.4.121 上取得，換版本會失去可對照性
!pip install -q ultralytics==8.4.121

import ultralytics
assert ultralytics.__version__ == "8.4.121", \
    f"ultralytics 版本不符：{ultralytics.__version__}"
print(f"▷ ultralytics {ultralytics.__version__}")

# Step.2 資料集準備
### 複製到可寫入工作區、校驗完整性、清 BOM 與舊快取、改寫 data.yaml 路徑。

In [ ]:
import os, shutil, sys, yaml

def render_progress_bar(current, total, task_name="檔案同步複製中", bar_length=25):
    percent = (current / total) * 100 if total > 0 else 100.0
    filled = int(bar_length * current // total) if total > 0 else bar_length
    bar = "█" * filled + "░" * (bar_length - filled)
    sys.stdout.write(f"\r▷ 正在執行 [{task_name}] | 進度: [{bar}] {percent:5.1f}% ({current}/{total})")
    sys.stdout.flush()


def copy_and_verify_dataset(src_dir, dst_dir):
    if not os.path.exists(src_dir):
        print(f"▷ 錯誤：找不到來源資料集目錄 {src_dir}")
        return False

    src_files = []
    for root, _, files in os.walk(src_dir):
        for file in files:
            src_files.append(os.path.relpath(os.path.join(root, file), src_dir))
    total_files = len(src_files)
    print(f"▷ 來源資料集掃描完成，共計 {total_files} 個檔案")

    for idx, rel_path in enumerate(src_files, 1):
        dst_path = os.path.join(dst_dir, rel_path)
        os.makedirs(os.path.dirname(dst_path), exist_ok=True)
        shutil.copy2(os.path.join(src_dir, rel_path), dst_path)
        if idx % 200 == 0 or idx == total_files:
            render_progress_bar(idx, total_files)

    print("\n\n▷ 正在檢查複製檔案")
    dst_files_set = set()
    for root, _, files in os.walk(dst_dir):
        for file in files:
            dst_files_set.add(os.path.relpath(os.path.join(root, file), dst_dir))

    missing, corrupted = [], []
    for rel_path in src_files:
        if rel_path not in dst_files_set:
            missing.append(rel_path)
        elif os.path.getsize(os.path.join(src_dir, rel_path)) != \
                os.path.getsize(os.path.join(dst_dir, rel_path)):
            corrupted.append(rel_path)

    print("≡" * 60)
    print("▷ 資料集複製完整性校驗：")
    print(f"  ▶ 來源檔案總數 : {len(src_files)}")
    print(f"  ▶ 目標檔案總數 : {len(dst_files_set)}")
    print(f"  ▶ 遺漏檔案數   : {len(missing)}")
    print(f"  ▶ 損毀/大小不符: {len(corrupted)}")
    ok = not missing and not corrupted
    print("▷ 檢查通過" if ok else f"▷ 檢查失敗  遺漏={missing[:5]}  損毀={corrupted[:5]}")
    print("≡" * 60)
    return ok


DST = "/kaggle/working/datasets-yolo26-v5-5"
DATA_YAML = "/kaggle/working/data.yaml"

assert copy_and_verify_dataset(DATASET_SRC, DST), "資料集複製失敗，不要往下跑"

# data.yaml 改寫成絕對路徑（來源版本用的是相對的 path: .）
with open(os.path.join(DST, "data.yaml"), encoding="utf-8") as f:
    ycfg = yaml.safe_load(f)
ycfg.update(path=DST, train="train/images", val="valid/images", test="test/images")
with open(DATA_YAML, "w", encoding="utf-8") as f:
    yaml.safe_dump(ycfg, f, default_flow_style=False, allow_unicode=True)

# v5.6 是 9 類。這裡刻意不寫死數字以外的假設：nc 由 data.yaml 決定，
# 後面的模型組態與驗證全部沿用 ycfg["nc"]。
NC = int(ycfg["nc"])
assert NC == 9, f"nc={NC}，v5.6 應為 9（8 = 舊的 v5r，跑錯資料集了）"
assert ycfg["names"][8] == "Thrips_Damage", f"第 9 類應為 Thrips_Damage，實際是 {ycfg['names'][8]}"

# BOM 與舊快取會讓 Ultralytics 的標註解析出錯
bom_fixed = cache_removed = 0
for subdir, _, files in os.walk(DST):
    for file in files:
        path = os.path.join(subdir, file)
        if file.endswith(".cache"):
            os.remove(path)
            cache_removed += 1
        elif file.endswith(".txt") and "labels" in subdir:
            with open(path, "rb") as fh:
                is_bom = fh.read(3) == b"\xef\xbb\xbf"
            if is_bom:
                with open(path, encoding="utf-8-sig") as fh:
                    content = fh.read()
                with open(path, "w", encoding="utf-8") as fh:
                    fh.write(content)
                bom_fixed += 1

print(f"▷ data.yaml → {DATA_YAML}   nc={NC}")
print(f"▷ names = {ycfg['names']}")
print(f"▷ 修正 BOM {bom_fixed} 個、清除快取 {cache_removed} 個")
print("▷ Step.2 完成")

# Step.3 模型組態
### 直接用 ultralytics 內建的 `yolo26-p2.yaml`，只覆寫 `nc`。不做任何模組替換。

In [ ]:
import yaml as _yaml
from pathlib import Path
from ultralytics.nn.tasks import DetectionModel
from ultralytics.utils.torch_utils import get_flops, get_num_params

# 讀已安裝的官方組態。v9 時代是先讀進來、再依臂別替換層；v10／v11 一層都不動。
_ULTRA = Path(ultralytics.__file__).parent
cfg = _yaml.safe_load((_ULTRA / "cfg/models/26/yolo26-p2.yaml").read_text(encoding="utf-8"))

cfg["nc"] = NC
cfg["scale"] = "n"
# end2end / reg_max 必須保留：YOLO26 靠這兩個 key 決定走 NMS-free 的 E2EDetectLoss
# （box + cls + l1）還是舊的 DFL(reg_max=16) 路徑。漏掉會靜默變成另一個體系的模型，
# 與 v8 / v9 完全無法對照。官方 yaml 本來就有，這裡只是明示不可被覆寫掉。
cfg.setdefault("end2end", True)
cfg.setdefault("reg_max", 1)

# 檔名必須帶 scale 字母：yaml_model_load 會用檔名覆寫 dict 裡的 scale 鍵，
# 少了 "n" 會落回 scales 字典的第一項並印出 warning。
YAML_PATH = "/kaggle/working/yolo26n-p2-v11.yaml"
with open(YAML_PATH, "w", encoding="utf-8") as f:
    _yaml.safe_dump(cfg, f, sort_keys=False, allow_unicode=True)

_m = DetectionModel(cfg=cfg, ch=3, nc=NC, verbose=False)
print(f"▷ v11: {get_num_params(_m):,} params / {get_flops(_m, imgsz=640):.2f} GFLOPs @640")
print(f"▷ yaml → {YAML_PATH}")
del _m

HP = dict(HYPERS)
HP.update(epochs=EPOCHS, patience=PATIENCE, close_mosaic=CLOSE_MOSAIC, name=RUN)
print("▷ 超參數：" + "  ".join(f"{k}={v}" for k, v in sorted(HP.items())))

# Step.4 架構隔離驗證
### 在燒 GPU 時數之前確認：體系正確、可前向、損失有限。

In [ ]:
import math, torch
from ultralytics import YOLO
from ultralytics.cfg import get_cfg
import ultralytics.utils.loss

model = YOLO(YAML_PATH)
core = model.model

# 1. E2E 體系必須與 v8 / v9 一致，否則走的是另一個損失路徑、完全無法對照
det = core.model[-1]
assert getattr(det, "end2end", False), "Detect 不是 end2end：yaml 少了 end2end: True"
assert det.reg_max == 1, f"reg_max={det.reg_max}，應為 1"
assert det.nc == NC and det.nl == 4, f"nc={det.nc} nl={det.nl}"
strides = [int(s) for s in det.stride]
assert strides == [4, 8, 16, 32], f"strides={strides}"
print(f"▷ 1/4 end2end=True, reg_max=1, nc={det.nc}, 偵測頭={det.nl}, strides={strides}")

# 2. 確認是乾淨的官方架構——v11 不該有任何自訂模組或被 patch 過的損失。
#    （這條在 v9 是「該有的模組要就位」，v10／v11 反過來：一個都不該有。）
types = {m.type.split(".")[-1] for m in core.model}
for forbidden in ("ADown", "StarTripletBlock"):
    assert forbidden not in types, f"不該出現 {forbidden}，v11 是原封不動的官方架構"
assert ultralytics.utils.loss.bbox_iou.__name__ == "bbox_iou", \
    f"損失函式被換過了：{ultralytics.utils.loss.bbox_iou.__name__}，v11 應為內建 CIoU"
print("▷ 2/4 無自訂模組、無 loss patch，確認為原封不動的官方 yolo26-p2")

# 3. 前向
core.eval()
with torch.no_grad():
    core(torch.zeros(1, 3, HP["imgsz"], HP["imgsz"]))
print(f"▷ 3/4 Forward pass ({HP['imgsz']}x{HP['imgsz']}) 成功")

# 4. 完整損失路徑，刻意用微小框貼近本資料集分佈
core.args = get_cfg(overrides={"box": HP["box"], "cls": HP["cls"], "dfl": HP["dfl"]})
core.train()
loss, items = core.loss({
    "img": torch.rand(2, 3, HP["imgsz"], HP["imgsz"]),
    "batch_idx": torch.tensor([0.0, 0.0, 1.0]),
    "cls": torch.tensor([[4.0], [8.0], [1.0]]),      # Scale_Insect / Thrips_Damage / Canker
    "bboxes": torch.tensor([[0.50, 0.50, 0.04, 0.04],
                            [0.22, 0.31, 0.20, 0.18],
                            [0.71, 0.68, 0.03, 0.03]]),
})
vals = ({k: float(v) for k, v in items.items()} if isinstance(items, dict)
        else {k: float(v) for k, v in zip(("box", "cls", "l1"), items.flatten())})
assert torch.isfinite(loss).all() and all(math.isfinite(v) for v in vals.values()), vals
print("▷ 4/4 損失路徑通過   " + "  ".join(f"{k}={v:.4f}" for k, v in vals.items()))
print("\n▷ Step.4 全部通過，可以開始訓練")

# Step.5 訓練

In [ ]:
import shutil, time
from ultralytics import YOLO

model = YOLO(YAML_PATH)
model.load("yolo26n.pt")     # v11 架構與官方一致，轉移率應為 902/902


def stop_and_snapshot(trainer):
    """每輪保留可續跑的 checkpoint，並在時數/輪數上限時乾淨停止。

    訓練迴圈結束後一定會執行 final_eval() → strip_optimizer()，把 last.pt / best.pt
    的 epoch 改成 -1 並清掉 optimizer/EMA，那種檔案無法續跑。
    on_fit_epoch_end 的觸發點在 save_model() 之後、跳出迴圈之前，此時 last.pt
    才剛寫好且尚未被 strip。

    備份不設條件：patience 早停時 trainer.stop 在進入這個 callback 之前就已為 True，
    若寫成 `if not trainer.stop` 這段會整個被跳過——那正是 v9 踩過的失效模式。
    """
    if trainer.last.exists():
        shutil.copy(trainer.last, trainer.wdir / "resume_from.pt")

    cap = STOP_AFTER_EPOCHS or trainer.epochs      # None → 跑滿，不提前停
    elapsed = (time.time() - trainer.train_time_start) / 3600
    if not trainer.stop and (trainer.epoch + 1 >= cap or elapsed > DEADLINE_HOURS):
        trainer.stop = True
        print(f"\n▷ 停於第 {trainer.epoch + 1} / {trainer.epochs} 輪，已耗時 {elapsed:.2f} h")
        print(f"▷ 續跑用 checkpoint：{trainer.wdir / 'resume_from.pt'}")


model.add_callback("on_fit_epoch_end", stop_and_snapshot)

_cap = STOP_AFTER_EPOCHS or EPOCHS
print(f"▷ 將跑到第 {_cap} / {EPOCHS} 輪"
      + ("" if _cap >= EPOCHS else "  ← STOP_AFTER_EPOCHS 會提前停止，剩餘輪次需用 RESUME.ipynb")
      + f"；牆鐘上限 {DEADLINE_HOURS} h")

results = model.train(data=DATA_YAML, save_period=10, **HP)
print("▷ v11 訓練完畢")

# Step.6 輸出整理

In [ ]:
import os, shutil

runs_dir = "/kaggle/working/runs"
if os.path.exists(runs_dir):
    print("▷ 正在壓縮訓練輸出")
    shutil.make_archive(f"/kaggle/working/runs_{RUN}", "zip", runs_dir)
    size = os.path.getsize(f"/kaggle/working/runs_{RUN}.zip") / (1024 ** 2)
    print(f"▷ 壓縮成功 /kaggle/working/runs_{RUN}.zip ({size:.2f} MB)")
else:
    print(f"▷ 壓縮失敗：找不到 {runs_dir}")

# Step.7 評估包
把後續分析要用的數字**存成檔案**（不只是印在畫面上），打包成一個幾十 KB 的 zip。
本機用 `tools/final_eval.py` / `tools/diag_localization.py` 做完整評估時可以對照。

In [ ]:
import csv, json, os, shutil, zipfile

def write_csv(path, fieldnames, rows):
    with open(path, "w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        w.writerows(rows)


# 用 best.pt 重跑一次驗證。訓練結束時 Model.train() 已把 self.model 換成 best.pt。
#
# plots=True 是必要的，不是為了畫圖：ultralytics 把 confusion_matrix.process_batch
# 包在 `if self.args.plots` 裡（detect/val.py:196），plots=False 會讓混淆矩陣維持全零。
m = model.val(data=DATA_YAML, imgsz=HP["imgsz"], batch=HP["batch"], plots=True)

RUN_DIR = str(model.trainer.save_dir)
OUT = f"/kaggle/working/eval_{RUN}"
os.makedirs(OUT, exist_ok=True)

# ── 1. 逐輪指標與超參數 ────────────────────────────────────────────
for fn in ("results.csv", "args.yaml"):
    src = os.path.join(RUN_DIR, fn)
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(OUT, fn))

# ── 2. 每類指標 ────────────────────────────────────────────────────
names = m.names if isinstance(m.names, dict) else {i: n for i, n in enumerate(m.names)}
COLS = ["class_id", "class", "precision", "recall", "f1", "ap50", "ap50_95"]
per_class = []
for i, ci in enumerate(m.box.ap_class_index):
    ci = int(ci)
    per_class.append({
        "class_id": ci,
        "class": names.get(ci, str(ci)),
        "precision": round(float(m.box.p[i]), 5),
        "recall": round(float(m.box.r[i]), 5),
        "f1": round(float(m.box.f1[i]), 5),
        "ap50": round(float(m.box.ap50[i]), 5),
        "ap50_95": round(float(m.box.ap[i]), 5),
    })
per_class.sort(key=lambda r: r["class_id"])
write_csv(os.path.join(OUT, "per_class.csv"), COLS, per_class)

# ── 3. 混淆矩陣（列=預測，欄=真實，最後一列/欄為背景）───────────────
cm = m.confusion_matrix.matrix
nc = len(names)
labels = [names.get(i, str(i)) for i in range(nc)] + ["background"]
with open(os.path.join(OUT, "confusion_matrix.csv"), "w", encoding="utf-8", newline="") as f:
    w = csv.writer(f)
    w.writerow([""] + [f"true_{l}" for l in labels])
    for i, lab in enumerate(labels):
        w.writerow([f"pred_{lab}"] + [int(cm[i][j]) for j in range(len(labels))])

fp_bg = {labels[i]: int(cm[i][nc]) for i in range(nc)}
fn_bg = {labels[i]: int(cm[nc][i]) for i in range(nc)}

# ── 4. F1-信心曲線與最佳截斷點 ──────────────────────────────────────
best_conf = None
try:
    x, y, _xl, _yl = m.curves_results[1]        # F1-Confidence(B)
    x = [float(v) for v in x]
    mean_f1 = ([sum(col) / len(col) for col in zip(*y)] if hasattr(y[0], "__len__")
               else [float(v) for v in y])
    with open(os.path.join(OUT, "f1_conf.csv"), "w", encoding="utf-8", newline="") as f:
        w = csv.writer(f)
        w.writerow(["conf", "mean_f1"])
        w.writerows(zip(x, mean_f1))
    best_conf = round(x[mean_f1.index(max(mean_f1))], 4)
except Exception as e:
    print(f"▷ F1-conf 曲線取用失敗（不影響主要結果）：{type(e).__name__}: {e}")

# ── 5. 平台期統計 ──────────────────────────────────────────────────
plateau = {}
rcsv = os.path.join(OUT, "results.csv")
if os.path.exists(rcsv):
    with open(rcsv, encoding="utf-8") as f:
        rec = [{k.strip(): v for k, v in row.items()} for row in csv.DictReader(f)]
    win = 50 if EPOCHS >= 120 else 16
    tail = rec[-win:]
    for key, col in (("mAP50", "metrics/mAP50(B)"), ("mAP50_95", "metrics/mAP50-95(B)")):
        if rec and col in rec[0]:
            vals = [float(r[col]) for r in tail]
            mean = sum(vals) / len(vals)
            var = sum((v - mean) ** 2 for v in vals) / max(len(vals) - 1, 1)
            plateau[key] = {"window": win, "mean": round(mean, 5),
                            "std": round(var ** 0.5, 5), "best": round(max(vals), 5)}

summary = {
    "run": RUN, "epochs": EPOCHS, "patience": PATIENCE, "close_mosaic": CLOSE_MOSAIC,
    "dataset": "v5.6", "nc": NC, "imgsz": HP["imgsz"],   # 執行當時此處寫死 v5.5，見第一個 cell
    "overall": {
        "mAP50": round(float(m.box.map50), 5),
        "mAP50_95": round(float(m.box.map), 5),
        "precision": round(float(m.box.mp), 5),
        "recall": round(float(m.box.mr), 5),
    },
    "plateau": plateau,
    "best_f1_conf": best_conf,
    "fp_from_background": fp_bg,
    "fn_to_background": fn_bg,
}
with open(os.path.join(OUT, "summary.json"), "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

shutil.make_archive(f"/kaggle/working/eval_{RUN}", "zip", OUT)
print(f"\n▷ 評估包 → /kaggle/working/eval_{RUN}.zip")
print(json.dumps(summary["overall"], ensure_ascii=False, indent=2))
if plateau:
    print("▷ 平台期：" + json.dumps(plateau, ensure_ascii=False))